In [1]:
# ============================================================
# FULLY STANDALONE POSITIVE DEFLECTION QC (PHY MAIN CHANNEL ONLY)
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------
# 1. Set paths
# -----------------------------
ks_dir = Path(r"C:\Users\Shermanlab\Desktop\2026-03-02_14-04-49\Record Node 101\experiment1\recording1\continuous\Neuropix-PXI-100.ProbeA\kilosort4")

# -----------------------------
# 2. Load required Kilosort/Phy files
# -----------------------------
templates = np.load(ks_dir / "templates.npy")              # (n_units, n_samples, n_channels)
channel_map = np.load(ks_dir / "channel_map.npy")          # physical channel ID for each template channel index
cluster_info = pd.read_csv(ks_dir / "cluster_info.tsv", sep="\t")

# -----------------------------
# 3. Build mappings
# -----------------------------
# cluster_id → Phy main channel ID
cluster_col = "cluster_id" if "cluster_id" in cluster_info.columns else "id"
channel_col = "ch" if "ch" in cluster_info.columns else "channel"

phy_best_ch_id = {
    int(row[cluster_col]): int(row[channel_col])
    for _, row in cluster_info.iterrows()
}

# physical channel ID → template channel index
phys_to_template_index = {phys_id: idx for idx, phys_id in enumerate(channel_map)}

# -----------------------------
# 4. Compute positive-bump ratio on Phy main channel only
# -----------------------------
unit_pos_ratio = {}
unit_pos_amp = {}
unit_neg_amp = {}

pos_ratio_threshold = 0.5

unit_ids = sorted(phy_best_ch_id.keys())   # ensures consistent ordering

for uid in unit_ids:

    main_ch_id = phy_best_ch_id[uid]

    # Skip if Phy's channel isn't in templates
    if main_ch_id not in phys_to_template_index:
        unit_pos_ratio[uid] = np.inf
        unit_pos_amp[uid] = 0.0
        unit_neg_amp[uid] = 0.0
        continue

    main_idx = phys_to_template_index[main_ch_id]

    # Extract waveform on Phy's exact channel
    trace = templates[uid][:, main_idx]

    neg = abs(np.min(trace))
    pos = abs(np.max(trace))

    unit_neg_amp[uid] = float(neg)
    unit_pos_amp[uid] = float(pos)

    if neg > 0:
        unit_pos_ratio[uid] = float(pos / neg)
    else:
        unit_pos_ratio[uid] = np.inf

# -----------------------------
# 5. Classification
# -----------------------------
pos_bad_units = [uid for uid in unit_ids if unit_pos_ratio[uid] >= pos_ratio_threshold]
pos_good_units = [uid for uid in unit_ids if unit_pos_ratio[uid] <  pos_ratio_threshold]

print("\n=== STANDALONE POSITIVE DEFLECTION QC (PHY MAIN CHANNEL ONLY) ===")
print(f"BAD units (pos/neg >= {pos_ratio_threshold}): {len(pos_bad_units)}")
print(f"GOOD units (pos/neg <  {pos_ratio_threshold}): {len(pos_good_units)}")

# Optional: print a few example ratios
print("\nExample ratios:")
for uid in unit_ids[:10]:
    print(f"UID {uid}: ratio={unit_pos_ratio[uid]:.3f}, neg={unit_neg_amp[uid]:.1f}, pos={unit_pos_amp[uid]:.1f}")



=== STANDALONE POSITIVE DEFLECTION QC (PHY MAIN CHANNEL ONLY) ===
BAD units (pos/neg >= 0.5): 179
GOOD units (pos/neg <  0.5): 249

Example ratios:
UID 0: ratio=1.430, neg=1.5, pos=2.2
UID 1: ratio=2.004, neg=0.6, pos=1.2
UID 2: ratio=1.015, neg=0.3, pos=0.3
UID 3: ratio=2.814, neg=0.4, pos=1.1
UID 4: ratio=0.312, neg=3.8, pos=1.2
UID 5: ratio=0.467, neg=2.1, pos=1.0
UID 6: ratio=3.389, neg=0.8, pos=2.8
UID 7: ratio=3.413, neg=0.8, pos=2.7
UID 8: ratio=0.323, neg=2.9, pos=0.9
UID 9: ratio=0.272, neg=2.5, pos=0.7
